In [1]:
from pathlib import Path
import sys
import subprocess
import time
from collections import deque
from datetime import datetime
from itertools import combinations
from typing import Optional

import numpy as np
import plotly.graph_objects as go
from prefect import flow, task
from prefect.context import get_run_context
from prefect_dask.task_runners import DaskTaskRunner

ROOT_CANDIDATES = [
    Path.cwd().resolve(),
    Path.cwd().resolve() / "main",
    Path.cwd().resolve().parent / "main",
]
for candidate in ROOT_CANDIDATES:
    if (candidate / "orchestrator").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from node_manager.patch_generator import (
    compute_patch_radii,
    generate_patch_centers,
    generate_patches_with_overlap,
)
from node_manager.mesh_builder import (
    triangulate_selected_nodes,
    mesh_quality_summary,
    save_mesh,
)
from quantum_processing.hamiltonian_builder import (
    compute_L,
    hamiltonian_builder,
    phi_circle_field_local,
)
from orchestrator.flow import (
    build_patch_records,
    persist_patch_metadata_task,
    merge_patches_gaussian_task,
    _qsim_ssh_command,
    _close_qsim_ssh_master,
    _ensure_hpc_remote_dirs,
    _stage_qaoa_hpc_hamiltonians,
    _poll_hpc_job_states,
    _list_remote_hpc_results,
    _fetch_remote_log_tail,
    fetch_hpc_qaoa_results_task,
)

RUNQAOA_SHELL = "~/qaoa_jobs/run_qaoa.sh"

In [214]:
n_nodes = 80
n_patch = 16
seed = 40
xy_min, xy_max = -1.0, 1.0
overlap_factor = 1.8
hamiltonian_concurrency = 16
qaoa_concurrency = 8
poll_interval_seconds = 15

hamiltonian_params = {
    "alpha": 10.0,
    "band_scale": 0.8,
    "gamma": 1,
    "use_sparsity": True,
    "sparsity_fraction": 0.8,
    "mu": 0.25,
    "use_repulsion": True,
    "d_min": 0.125,
    "eta": 0.8,
    "use_bend": True,
    "kappa": 3.0,
    "use_max_edge": True,
    "d_max_scale": 1.2,
    "eta_max": 40.0,
    "use_density_field": True,
    "density_radius_scale": 0.8,
    "gamma_density": 20.0,
    "use_angular_bins": True,
    "num_angular_bins": 6,
    "eta_theta": 20.0,
    "use_collinearity_penalty": True,
    "eta_col": 20.0,
    "beta": 50.0,
    "normalize": True,
    "tuning_factors": {
        "domain": 0.8,
        "spacing": 0.3,
        "sparsity": 1.0,
        "bend": 1.3,
        "max_edge": 1.2,
        "density": 0.4,
        "angular_bins": 1.7,
        "collinearity": 1.8,
        "boundary_alignment": 0,
    },
}

In [215]:
@task
def generate_random_2d_nodes_task(n_nodes, seed=42, xy_min=-1.0, xy_max=1.0):
    rng = np.random.default_rng(seed)
    nodes = rng.uniform(xy_min, xy_max, size=(int(n_nodes), 2))
    return nodes


@task
def generate_2d_patches_task(nodes, n_patch, overlap_factor=1.25):
    L = compute_L(nodes)
    r_patch, r_halo, d_min = compute_patch_radii(
        nodes,
        L=L,
        Q_max=int(n_patch),
    )

    centers = generate_patch_centers(nodes, r_patch)
    if len(centers) == 0:
        centers = np.asarray([np.asarray(nodes, dtype=float).mean(axis=0)])

    patches = generate_patches_with_overlap(
        nodes=nodes,
        centers=centers,
        r_patch=r_patch,
        r_halo=r_halo,
        Q_max=int(n_patch),
        overlap_factor=overlap_factor,
        cad_boundary_idx=None,
    )

    all_indices = np.arange(len(nodes), dtype=np.intp)
    covered = set()
    for patch in patches:
        interior = np.asarray(patch.get("interior_idx", []), dtype=np.intp)
        halo = np.asarray(patch.get("halo_idx", []), dtype=np.intp)
        covered.update(interior.tolist())
        covered.update(halo.tolist())

    uncovered = np.setdiff1d(
        all_indices,
        np.asarray(sorted(covered), dtype=np.intp),
        assume_unique=False,
    )
    fallback_patch_count = 0
    remaining_uncovered = np.asarray(uncovered, dtype=np.intp)
    while len(remaining_uncovered) > 0:
        center_idx = int(remaining_uncovered[0])
        distances = np.linalg.norm(nodes[remaining_uncovered] - nodes[center_idx], axis=1)
        local_uncovered = remaining_uncovered[np.argsort(distances)[:min(int(n_patch), len(remaining_uncovered))]]
        patches.append({
            "center": nodes[center_idx],
            "interior_idx": np.asarray(local_uncovered, dtype=np.intp),
            "halo_idx": np.asarray([], dtype=np.intp),
            "patch_id": len(patches),
            "cad_boundary_idx_local": None,
            "region_type": "critical",
        })
        fallback_patch_count += 1
        remaining_uncovered = np.setdiff1d(remaining_uncovered, local_uncovered, assume_unique=False)

    for patch in patches:
        patch["region_type"] = "critical"

    print(f"Random 2D nodes: {len(nodes)}")
    print(f"Patch node cap n_patch: {int(n_patch)}")
    print(f"Generated patches: {len(patches)}")
    print(f"Characteristic L from nodes: {L:.6f}")
    print(f"Computed r_patch: {r_patch:.6f}")
    print(f"Computed r_halo: {r_halo:.6f}")
    print(f"Computed d_min: {d_min:.6f}")
    print(f"Patch centers before coverage fallback: {len(centers)}")
    print(f"Fallback patches added: {fallback_patch_count}")
    return patches, r_patch


In [216]:
@task
def build_2d_hamiltonian_task(record, ham_dir: str, rec_dir: str, params: dict):
    L = compute_L(record.patch_nodes)
    center = np.asarray(record.patch_nodes).mean(axis=0)
    dists = np.linalg.norm(np.asarray(record.patch_nodes) - center, axis=1)
    R = np.percentile(dists, 90)
    phi = phi_circle_field_local(record.patch_nodes, R=R)
    band = float(params["band_scale"]) * R

    has_boundary = (
        record.boundary_nodes_idx is not None
        and len(record.boundary_nodes_idx) > 0
    )
    boundary_nodes = record.boundary_nodes_idx if has_boundary else None

    H = hamiltonian_builder(
        phi=phi,
        r=record.patch_nodes,
        alpha=params["alpha"],
        band=band,
        gamma=params["gamma"],
        use_sparsity=params["use_sparsity"],
        N=int(float(params["sparsity_fraction"]) * len(phi)),
        mu=params["mu"],
        use_repulsion=params["use_repulsion"],
        d_min=params["d_min"],
        eta=params["eta"],
        use_bend=params["use_bend"],
        kappa=params["kappa"],
        use_max_edge=params["use_max_edge"],
        d_max=float(params["d_max_scale"]) * L,
        eta_max=params["eta_max"],
        use_density_field=params["use_density_field"],
        density_radius=float(params["density_radius_scale"]) * L,
        gamma_density=params["gamma_density"],
        use_angular_bins=params["use_angular_bins"],
        num_angular_bins=params["num_angular_bins"],
        eta_theta=params["eta_theta"],
        use_collinearity_penalty=params["use_collinearity_penalty"],
        eta_col=params["eta_col"],
        use_boundary_alignment=has_boundary,
        boundary_nodes=boundary_nodes,
        beta=params["beta"],
        normalize=params["normalize"],
        tuning_factors=params["tuning_factors"],
    )

    record.phi = phi
    ham_path = Path(ham_dir) / f"{record.patch_id}.npz"

    sparse_terms = H.to_sparse_list()
    sparse_ops = np.asarray([term[0] for term in sparse_terms], dtype="U2")
    sparse_positions = np.full((len(sparse_terms), 2), -1, dtype=np.int32)
    sparse_coeffs = np.asarray([complex(term[2]) for term in sparse_terms], dtype=np.complex128)
    for idx, (_, positions, _) in enumerate(sparse_terms):
        sparse_positions[idx, :len(positions)] = np.asarray(positions, dtype=np.int32)

    np.savez(
        ham_path,
        sparse_ops=sparse_ops,
        sparse_positions=sparse_positions,
        coeffs=sparse_coeffs,
        num_qubits=np.asarray([H.num_qubits], dtype=np.int32),
    )

    record.hamiltonian_path = str(ham_path)
    record.save(rec_dir)
    return record

In [217]:
@task(name="send_2d_hpc_qaoa_wave")
def send_2d_hpc_qaoa_wave_task(
    wave_records,
    remote_run_dir: str,
    remote_log_dir: str,
    local_hpc_result_dir: str,
):
    local_hpc_result_dir_path = Path(local_hpc_result_dir)

    _stage_qaoa_hpc_hamiltonians(
        wave_records,
        remote_run_dir=remote_run_dir,
    )

    handles = []
    for record in wave_records:
        patch_id = record.patch_id
        remote_ham = f"{remote_run_dir}/{patch_id}.npz"
        remote_out = f"{remote_run_dir}/{patch_id}.json"
        submit_cmd = (
            f"sbatch -o /dev/null -e /dev/null {RUNQAOA_SHELL} "
            f"{remote_ham} {remote_out} {remote_log_dir}"
        )
        result = subprocess.check_output(
            _qsim_ssh_command(submit_cmd),
            text=True,
        ).strip()
        job_id = result.split()[-1]
        handles.append({
            "patch_id": patch_id,
            "job_id": job_id,
            "remote_out": remote_out,
            "remote_log": f"{remote_log_dir}/slurm-{job_id}.out",
            "local_out": str(local_hpc_result_dir_path / f"{patch_id}.json"),
            "missing_result_polls": 0,
        })

    return handles


def run_2d_qaoa_hpc_batch(
    records,
    local_result_dir: str,
    remote_run_dir: str,
    remote_log_dir: str,
    poll_interval_seconds: int = 15,
    max_outstanding_jobs: int = 8,
):
    records = list(records)
    if not records:
        return []

    poll_interval_seconds = max(1, int(poll_interval_seconds))
    max_outstanding_jobs = max(1, int(max_outstanding_jobs))

    local_hpc_result_dir = Path(local_result_dir) / "hpc_results"
    local_hpc_result_dir.mkdir(parents=True, exist_ok=True)

    _ensure_hpc_remote_dirs(remote_run_dir, remote_log_dir)

    pending_records = deque(records)
    active_jobs = {}
    completed_records = {}
    records_by_patch_id = {record.patch_id: record for record in records}

    while pending_records or active_jobs:
        if pending_records and len(active_jobs) < max_outstanding_jobs:
            wave_records = []
            while pending_records and len(active_jobs) + len(wave_records) < max_outstanding_jobs:
                wave_records.append(pending_records.popleft())

            submitted_handles = send_2d_hpc_qaoa_wave_task(
                wave_records,
                remote_run_dir=remote_run_dir,
                remote_log_dir=remote_log_dir,
                local_hpc_result_dir=str(local_hpc_result_dir),
            )
            for handle in submitted_handles:
                active_jobs[handle["job_id"]] = handle

        active_states = _poll_hpc_job_states(active_jobs.keys())
        available_results = _list_remote_hpc_results(remote_run_dir)

        ready_to_fetch = []
        for job_id, handle in list(active_jobs.items()):
            remote_filename = Path(handle["remote_out"]).name
            if job_id in active_states:
                continue

            if remote_filename in available_results:
                ready_to_fetch.append(handle)
                active_jobs.pop(job_id)
                continue

            handle["missing_result_polls"] += 1
            if handle["missing_result_polls"] >= 3:
                log_tail = _fetch_remote_log_tail(handle["remote_log"])
                raise FileNotFoundError(
                    f"No HPC result file found for patch {handle['patch_id']} / job {job_id}. "
                    f"Expected result file {handle['remote_out']} and log file {handle['remote_log']}.\n\n"
                    f"--- Tail of Slurm log ---\n{log_tail}"
                )

        if ready_to_fetch:
            fetched_results = fetch_hpc_qaoa_results_task(ready_to_fetch)
            for result in fetched_results:
                record = records_by_patch_id[result["patch_id"]]
                record.bitstring = result["bitstring"]
                record.energy = result["energy"]
                record.save(local_result_dir)
                completed_records[record.patch_id] = record

        if active_jobs:
            time.sleep(poll_interval_seconds)

    _close_qsim_ssh_master()

    return [
        completed_records[record.patch_id]
        for record in records
        if record.patch_id in completed_records
    ]

In [218]:
@task
def build_2d_mesh_task(
    nodes,
    merged_indices,
    output_dir,
    formats=("msh", "vtk", "obj"),
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    merged_indices = np.unique(np.asarray(merged_indices, dtype=np.intp))
    if len(merged_indices) < 3:
        raise ValueError(f"Need at least 3 selected nodes for 2D triangulation, got {len(merged_indices)}")

    mesh_nodes, triangles, global_idx = triangulate_selected_nodes(
        nodes,
        merged_indices,
        polygons=None,
    )

    quality = mesh_quality_summary(mesh_nodes, triangles) if len(triangles) > 0 else {}
    files = []
    for fmt in formats:
        files.append(save_mesh(
            mesh_nodes,
            triangles,
            output_dir / f"optimised_2d_mesh.{fmt}",
            fmt=fmt,
            quality=quality,
        ))

    np.save(output_dir / "mesh_nodes.npy", mesh_nodes)
    np.save(output_dir / "triangles.npy", triangles)
    np.save(output_dir / "global_idx.npy", global_idx)

    print("\n2D mesh created")
    print(f"  Nodes: {quality.get('n_nodes', '?')}")
    print(f"  Triangles: {quality.get('n_elements', '?')}")
    print(f"  Min angle: {quality.get('min_angle', '?'):.2f}")
    print(f"  Mean min angle: {quality.get('mean_min_angle', '?'):.2f}")
    print(f"  Mean aspect ratio: {quality.get('mean_aspect_ratio', '?'):.3f}")
    print(f"  Mean skewness: {quality.get('mean_skewness', '?'):.4f}")

    return {
        "nodes": mesh_nodes,
        "triangles": triangles,
        "global_idx": global_idx,
        "quality": quality,
        "files": files,
    }

In [219]:
@flow(task_runner=DaskTaskRunner(
    cluster_kwargs={
        "n_workers": 10,
        "threads_per_worker": 2,
        "processes": True,
        "memory_limit": "auto",
        "timeout": "120s",
        "death_timeout": "240s",
    }
))
def random_2d_hpc_mesh_pipeline(
    n_nodes: int = 40,
    n_patch: int = 14,
    seed: int = 42,
    xy_min: float = -1.0,
    xy_max: float = 1.0,
    overlap_factor: float = 1.25,
    hamiltonian_concurrency: int = 16,
    qaoa_concurrency: int = 8,
    poll_interval_seconds: int = 15,
    hamiltonian_params: Optional[dict] = None,
    remote_run_dir: Optional[str] = None,
    remote_log_dir: Optional[str] = None,
):
    ctx = get_run_context()
    run_id = str(ctx.flow_run.id)
    base_dir = Path("outputs") / f"2d_hpc_{run_id}"
    ham_dir = base_dir / "hamiltonians"
    rec_dir = base_dir / "records"
    mesh_dir = base_dir / "mesh"
    ham_dir.mkdir(parents=True, exist_ok=True)
    rec_dir.mkdir(parents=True, exist_ok=True)

    remote_run_name = f"2d_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{run_id}"
    if remote_run_dir is None:
        remote_run_dir = f"~/hpc_runs/{remote_run_name}"
    if remote_log_dir is None:
        remote_log_dir = f"~/qaoa_logs/{remote_run_name}"
    if hamiltonian_params is None:
        hamiltonian_params = globals()["hamiltonian_params"]

    nodes = generate_random_2d_nodes_task(
        n_nodes=n_nodes,
        seed=seed,
        xy_min=xy_min,
        xy_max=xy_max,
    )
    patches, r_patch = generate_2d_patches_task(
        nodes,
        n_patch=n_patch,
        overlap_factor=overlap_factor,
    )
    records = build_patch_records(nodes, patches)
    persist_patch_metadata_task(records, str(rec_dir))

    print(f"QAOA shell: {RUNQAOA_SHELL}")
    print(f"Remote HPC run dir: {remote_run_dir}")
    print(f"Remote HPC log dir: {remote_log_dir}")

    ham_futures = deque()
    built_records = []
    for record in records:
        ham_futures.append(build_2d_hamiltonian_task.submit(
            record,
            str(ham_dir),
            str(rec_dir),
            hamiltonian_params,
        ))
        if len(ham_futures) >= int(hamiltonian_concurrency):
            built_records.append(ham_futures.popleft().result())

    while ham_futures:
        built_records.append(ham_futures.popleft().result())

    qaoa_records = run_2d_qaoa_hpc_batch(
        built_records,
        str(rec_dir),
        remote_run_dir=remote_run_dir,
        remote_log_dir=remote_log_dir,
        poll_interval_seconds=poll_interval_seconds,
        max_outstanding_jobs=qaoa_concurrency,
    )

    merged_indices = merge_patches_gaussian_task(qaoa_records, nodes, r_patch)
    merged_indices = np.asarray(merged_indices, dtype=np.intp)
    np.save(base_dir / "merged_indices.npy", merged_indices)

    mesh_info = build_2d_mesh_task(
        nodes,
        merged_indices,
        str(mesh_dir),
        formats=("msh", "vtk", "obj"),
    )

    return {
        "base_dir": str(base_dir),
        "nodes": nodes,
        "patches": patches,
        "records": qaoa_records,
        "merged_indices": merged_indices,
        "mesh_info": mesh_info,
        "r_patch": r_patch,
    }

In [220]:
result = random_2d_hpc_mesh_pipeline(
    n_nodes=n_nodes,
    n_patch=n_patch,
    seed=seed,
    xy_min=xy_min,
    xy_max=xy_max,
    overlap_factor=overlap_factor,
    hamiltonian_concurrency=hamiltonian_concurrency,
    qaoa_concurrency=qaoa_concurrency,
    poll_interval_seconds=poll_interval_seconds,
    hamiltonian_params=hamiltonian_params,
)
result["base_dir"]

16:47:36.341 | INFO    | Flow run 'nano-tuna' - Beginning flow run 'nano-tuna' for flow 'random-2d-hpc-mesh-pipeline'

16:47:36.348 | INFO    | Flow run 'nano-tuna' - View at http://127.0.0.1:4200/runs/flow-run/452ff809-4cfe-4ffe-93e1-8502deb37216

16:47:36.359 | INFO    | Task run 'generate_random_2d_nodes_task-846' - Finished in state Completed()

Random 2D nodes: 80
Patch node cap n_patch: 16
Generated patches: 6
Characteristic L from nodes: 0.122925
Computed r_patch: 0.478662
Computed r_halo: 0.675343
Computed d_min: 0.098340
Patch centers before coverage fallback: 4
Fallback patches added: 2


16:47:36.370 | INFO    | Task run 'generate_2d_patches_task-b00' - Finished in state Completed()

...

16:50:09.256 | INFO    | distributed.nanny - Nanny at 'tcp://127.0.0.1:42781' closed.

16:50:09.258 | INFO    | distributed.scheduler - Closing scheduler. Reason: unknown

16:50:09.259 | INFO    | distributed.scheduler - Scheduler closing all comms

16:50:09.261 | INFO    | Flow run 'nano-tuna' - Finished in state Completed()

'outputs/2d_hpc_452ff809-4cfe-4ffe-93e1-8502deb37216'

In [221]:
nodes = np.asarray(result["nodes"], dtype=float)
patches = result["patches"]
records = result["records"]
merged_indices = np.asarray(result["merged_indices"], dtype=np.intp)
mesh_info = result["mesh_info"]
mesh_nodes = np.asarray(mesh_info["nodes"], dtype=float)
triangles = np.asarray(mesh_info["triangles"], dtype=np.intp)
global_idx = np.asarray(mesh_info["global_idx"], dtype=np.intp)

fig = go.Figure()
fig.add_trace(go.Scattergl(
    x=nodes[:, 0],
    y=nodes[:, 1],
    mode="markers+text",
    marker=dict(size=7, color="black", opacity=0.65),
    text=[str(i) for i in range(len(nodes))],
    textposition="top center",
    hovertemplate="node %{text}<extra></extra>",
    name="Random nodes",
))

for i, patch in enumerate(patches):
    interior = np.asarray(patch.get("interior_idx", []), dtype=np.intp)
    halo = np.asarray(patch.get("halo_idx", []), dtype=np.intp)
    patch_idx = np.concatenate([interior, halo]) if halo.size else interior
    if patch_idx.size == 0:
        continue
    pts = nodes[patch_idx]
    fig.add_trace(go.Scattergl(
        x=pts[:, 0],
        y=pts[:, 1],
        mode="markers",
        marker=dict(size=10, opacity=0.28),
        name=f"patch {i}",
        hovertemplate=f"patch {i}<extra></extra>",
        showlegend=False,
    ))

fig.add_trace(go.Scattergl(
    x=nodes[merged_indices, 0],
    y=nodes[merged_indices, 1],
    mode="markers",
    marker=dict(size=12, color="#d62728", symbol="circle-open", line=dict(width=2)),
    name="Merged QAOA nodes",
))

fig.update_layout(
    title=f"Random 2D Nodes, Patches, and Merged Selection (N={len(nodes)}, n_patch={n_patch})",
    xaxis_title="x",
    yaxis_title="y",
    template="plotly_white",
    width=950,
    height=760,
)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()

In [222]:
edge_x = []
edge_y = []
for tri in triangles:
    pts = mesh_nodes[tri]
    closed = np.vstack([pts, pts[0]])
    edge_x.extend(closed[:, 0].tolist() + [None])
    edge_y.extend(closed[:, 1].tolist() + [None])

fig_mesh = go.Figure()
fig_mesh.add_trace(go.Scattergl(
    x=edge_x,
    y=edge_y,
    mode="lines",
    line=dict(color="rgba(40,40,40,0.45)", width=1.5),
    name="Triangle edges",
))
fig_mesh.add_trace(go.Scattergl(
    x=mesh_nodes[:, 0],
    y=mesh_nodes[:, 1],
    mode="markers",
    marker=dict(size=7, color="#1f77b4"),
    name="Mesh nodes",
))
fig_mesh.update_layout(
    title="2D Triangulated Mesh After Patch Merge",
    xaxis_title="x",
    yaxis_title="y",
    template="plotly_white",
    width=950,
    height=760,
)
fig_mesh.update_yaxes(scaleanchor="x", scaleratio=1)
fig_mesh.show()

In [223]:
def triangle_quality_metrics(points):
    pts = np.asarray(points, dtype=float)
    edge_lengths = np.array([
        np.linalg.norm(pts[i] - pts[j])
        for i, j in combinations(range(3), 2)
    ], dtype=float)

    area = 0.5 * abs(np.cross(pts[1] - pts[0], pts[2] - pts[0]))
    min_edge = float(np.min(edge_lengths))
    max_edge = float(np.max(edge_lengths))
    aspect_ratio = max_edge / max(min_edge, 1e-12)

    angles = []
    for i in range(3):
        v1 = pts[(i + 1) % 3] - pts[i]
        v2 = pts[(i + 2) % 3] - pts[i]
        denom = np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-30
        cos_a = np.dot(v1, v2) / denom
        angles.append(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))
    angles = np.asarray(angles, dtype=float)

    theta_min = float(angles.min())
    theta_max = float(angles.max())
    skewness = float(max((theta_max - 60.0) / 120.0, (60.0 - theta_min) / 60.0))
    mean_ratio_quality = float(np.clip(1.0 - skewness, 0.0, 1.0))

    return {
        "area": float(area),
        "min_edge": min_edge,
        "max_edge": max_edge,
        "aspect_ratio": float(aspect_ratio),
        "min_angle": theta_min,
        "max_angle": theta_max,
        "mean_ratio_quality": mean_ratio_quality,
        "skewness": skewness,
    }


quality_rows = []
centroids = []
for tri_id, tri in enumerate(triangles):
    tri_points = mesh_nodes[tri]
    metrics = triangle_quality_metrics(tri_points)
    metrics["triangle_id"] = int(tri_id)
    metrics["local_indices"] = tri.tolist()
    metrics["global_indices"] = global_idx[tri].tolist()
    quality_rows.append(metrics)
    centroids.append(tri_points.mean(axis=0))

centroids = np.asarray(centroids, dtype=float)
metric_names = ["area", "aspect_ratio", "min_angle", "mean_ratio_quality", "skewness"]
summary_metric = []
summary_min = []
summary_mean = []
summary_max = []
for name in metric_names:
    values = np.array([row[name] for row in quality_rows], dtype=float)
    summary_metric.append(name)
    summary_min.append(f"{values.min():.6f}")
    summary_mean.append(f"{values.mean():.6f}")
    summary_max.append(f"{values.max():.6f}")

fig_quality_table = go.Figure(data=[go.Table(
    header=dict(
        values=["Metric", "Min", "Mean", "Max"],
        fill_color="#1f2937",
        font=dict(color="white"),
    ),
    cells=dict(
        values=[summary_metric, summary_min, summary_mean, summary_max],
        fill_color="#f9fafb",
    ),
)])
fig_quality_table.update_layout(title="2D Triangle Mesh Quality Summary", width=720, height=360)
fig_quality_table.show()

hover_text = [
    "<br>".join([
        f"triangle {row['triangle_id']}",
        f"global indices: {row['global_indices']}",
        f"area: {row['area']:.6f}",
        f"aspect_ratio: {row['aspect_ratio']:.6f}",
        f"min_angle: {row['min_angle']:.6f}",
        f"mean_ratio_quality: {row['mean_ratio_quality']:.6f}",
        f"skewness: {row['skewness']:.6f}",
    ])
    for row in quality_rows
]

fig_quality = go.Figure()
fig_quality.add_trace(go.Scattergl(
    x=edge_x,
    y=edge_y,
    mode="lines",
    name="",
    line=dict(color="rgba(70,70,70,0.35)", width=1.5),
))
fig_quality.add_trace(go.Scattergl(
    x=centroids[:, 0],
    y=centroids[:, 1],
    mode="markers+text",
    name="",
    marker=dict(
        size=10,
        color=[row["skewness"] for row in quality_rows],
        colorscale="Turbo",
        cmin=0.0,
        cmax=1.0,
        colorbar=dict(title="skewness"),
        opacity=0.9,
    ),
    text=[f"tri {row['triangle_id']}" for row in quality_rows],
    textposition="bottom center",
    hovertext=hover_text,
    hovertemplate="%{hovertext}<extra></extra>",
))
fig_quality.update_layout(
    title="2D Triangle Element Quality Visualizer",
    xaxis_title="x",
    yaxis_title="y",
    width=1050,
    height=800,
    template="plotly_white",
)
fig_quality.update_yaxes(scaleanchor="x", scaleratio=1)
fig_quality.show()

quality_rows

/tmp/ipykernel_635553/3336174539.py:8: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  area = 0.5 * abs(np.cross(pts[1] - pts[0], pts[2] - pts[0]))


[{'area': 0.12590636159576835,
  'min_edge': 0.4828662214254055,
  'max_edge': 0.5849495804740668,
  'aspect_ratio': 1.2114112657276264,
  'min_angle': 49.61155736838473,
  'max_angle': 67.3232247506662,
  'mean_ratio_quality': 0.8268592894730789,
  'skewness': 0.17314071052692118,
  'triangle_id': 0,
  'local_indices': [17, 5, 16],
  'global_indices': [70, 27, 68]},
 {'area': 0.06829408307386584,
  'min_edge': 0.4828662214254055,
  'max_edge': 1.2097981119911727,
  'aspect_ratio': 2.505451941574807,
  'min_angle': 8.671055422784711,
  'max_angle': 157.8071322036762,
  'mean_ratio_quality': 0.1445175903797452,
  'skewness': 0.8554824096202548,
  'triangle_id': 1,
  'local_indices': [5, 17, 6],
  'global_indices': [27, 70, 33]},
 {'area': 0.03638660565109961,
  'min_edge': 0.3834102142524921,
  'max_edge': 1.1555328637388258,
  'aspect_ratio': 3.0138291072700993,
  'min_angle': 4.631895110482055,
  'max_angle': 165.91399331618953,
  'mean_ratio_quality': 0.07719825184136764,
  'skewness

In [224]:
def _triangle_quality_metrics_local(points):
    pts = np.asarray(points, dtype=float)
    edge_lengths = np.array([
        np.linalg.norm(pts[i] - pts[j])
        for i, j in combinations(range(3), 2)
    ], dtype=float)

    area = 0.5 * abs(np.cross(pts[1] - pts[0], pts[2] - pts[0]))
    min_edge = float(np.min(edge_lengths))
    max_edge = float(np.max(edge_lengths))
    aspect_ratio = max_edge / max(min_edge, 1e-12)

    angles = []
    for i in range(3):
        v1 = pts[(i + 1) % 3] - pts[i]
        v2 = pts[(i + 2) % 3] - pts[i]
        denom = np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-30
        cos_a = np.dot(v1, v2) / denom
        angles.append(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))
    angles = np.asarray(angles, dtype=float)

    theta_min = float(angles.min())
    theta_max = float(angles.max())
    skewness = float(max((theta_max - 60.0) / 120.0, (60.0 - theta_min) / 60.0))
    mean_ratio_quality = float(np.clip(1.0 - skewness, 0.0, 1.0))

    return {
        "area": float(area),
        "min_edge": float(min_edge),
        "max_edge": float(max_edge),
        "aspect_ratio": float(aspect_ratio),
        "min_angle": theta_min,
        "max_angle": theta_max,
        "mean_ratio_quality": mean_ratio_quality,
        "skewness": skewness,
    }


def _quality_rows_for_mesh(mesh_nodes, mesh_triangles, mesh_global_idx):
    rows = []
    centroids = []
    for tri_id, tri in enumerate(np.asarray(mesh_triangles, dtype=np.intp)):
        tri_points = mesh_nodes[tri]
        metrics = _triangle_quality_metrics_local(tri_points)
        metrics["triangle_id"] = int(tri_id)
        metrics["local_indices"] = tri.tolist()
        metrics["global_indices"] = np.asarray(mesh_global_idx, dtype=np.intp)[tri].tolist()
        rows.append(metrics)
        centroids.append(tri_points.mean(axis=0))
    return rows, np.asarray(centroids, dtype=float)


def greedy_farthest_point_node_selection(nodes, target_count):
    pts = np.asarray(nodes, dtype=float)
    target_count = int(min(max(3, target_count), len(pts)))

    center = pts.mean(axis=0)
    first = int(np.argmin(np.linalg.norm(pts - center, axis=1)))
    selected = [first]
    remaining = np.ones(len(pts), dtype=bool)
    remaining[first] = False

    while len(selected) < target_count:
        selected_pts = pts[np.asarray(selected, dtype=np.intp)]
        distances_to_selected = np.linalg.norm(
            pts[:, None, :] - selected_pts[None, :, :],
            axis=2,
        ).min(axis=1)
        distances_to_selected[~remaining] = -np.inf
        next_idx = int(np.argmax(distances_to_selected))
        selected.append(next_idx)
        remaining[next_idx] = False

    return np.asarray(selected, dtype=np.intp)


nodes = np.asarray(result["nodes"], dtype=float)
qaoa_indices = np.asarray(result["merged_indices"], dtype=np.intp)
greedy_indices = greedy_farthest_point_node_selection(nodes, target_count=len(qaoa_indices))

greedy_mesh_nodes, greedy_triangles, greedy_global_idx = triangulate_selected_nodes(
    nodes,
    greedy_indices,
    polygons=None,
)
greedy_quality = mesh_quality_summary(greedy_mesh_nodes, greedy_triangles)
greedy_quality_rows, greedy_centroids = _quality_rows_for_mesh(
    greedy_mesh_nodes,
    greedy_triangles,
    greedy_global_idx,
)

qaoa_mesh_info = result["mesh_info"]
qaoa_mesh_nodes = np.asarray(qaoa_mesh_info["nodes"], dtype=float)
qaoa_triangles = np.asarray(qaoa_mesh_info["triangles"], dtype=np.intp)
qaoa_global_idx = np.asarray(qaoa_mesh_info["global_idx"], dtype=np.intp)
qaoa_quality = qaoa_mesh_info["quality"]
qaoa_quality_rows, qaoa_centroids = _quality_rows_for_mesh(
    qaoa_mesh_nodes,
    qaoa_triangles,
    qaoa_global_idx,
)

greedy_dir = Path(result["base_dir"]) / "greedy_mesh"
greedy_dir.mkdir(parents=True, exist_ok=True)
for fmt in ("msh", "vtk", "obj"):
    save_mesh(
        greedy_mesh_nodes,
        greedy_triangles,
        greedy_dir / f"greedy_2d_mesh.{fmt}",
        fmt=fmt,
        quality=greedy_quality,
    )
np.save(greedy_dir / "greedy_indices.npy", greedy_indices)
np.save(greedy_dir / "mesh_nodes.npy", greedy_mesh_nodes)
np.save(greedy_dir / "triangles.npy", greedy_triangles)
np.save(greedy_dir / "global_idx.npy", greedy_global_idx)

print(f"QAOA selected nodes: {len(qaoa_indices)}")
print(f"Greedy selected nodes: {len(greedy_indices)}")
print(f"Greedy mesh saved to: {greedy_dir}")
greedy_quality

QAOA selected nodes: 18
Greedy selected nodes: 18
Greedy mesh saved to: outputs/2d_hpc_452ff809-4cfe-4ffe-93e1-8502deb37216/greedy_mesh


/tmp/ipykernel_635553/1271520833.py:8: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  area = 0.5 * abs(np.cross(pts[1] - pts[0], pts[2] - pts[0]))


{'n_nodes': 18,
 'n_elements': 29,
 'min_angle': 0.0189771059222601,
 'mean_min_angle': 32.17324089291109,
 'max_aspect_ratio': 3.965570601529101,
 'mean_aspect_ratio': 1.7947528761060796,
 'mean_skewness': 0.46377931845148185,
 'max_skewness': 0.9996837149012957}

In [225]:
from pathlib import Path
from datetime import datetime

from plotly.subplots import make_subplots


# ============================================================
# Create timestamped visualization output directory
# ============================================================



# ============================================================
# Helper functions
# ============================================================

def _mesh_edges_xy(mesh_nodes, mesh_triangles):
    edge_x = []
    edge_y = []

    for tri in np.asarray(mesh_triangles, dtype=np.intp):
        pts = mesh_nodes[tri]
        closed = np.vstack([pts, pts[0]])

        edge_x.extend(closed[:, 0].tolist() + [None])
        edge_y.extend(closed[:, 1].tolist() + [None])

    return edge_x, edge_y


def _quality_summary_from_rows(rows, mesh_quality):
    return {
        "n_nodes": float(mesh_quality["n_nodes"]),
        "n_elements": float(mesh_quality["n_elements"]),
        "min_angle": float(mesh_quality["min_angle"]),
        "mean_min_angle": float(mesh_quality["mean_min_angle"]),
        "mean_aspect_ratio": float(mesh_quality["mean_aspect_ratio"]),
        "max_aspect_ratio": float(mesh_quality["max_aspect_ratio"]),
        "mean_skewness": float(mesh_quality["mean_skewness"]),
        "max_skewness": float(mesh_quality["max_skewness"]),
        "mean_ratio_quality": float(
            np.mean([row["mean_ratio_quality"] for row in rows])
        ),
    }


# ============================================================
# Mesh quality summaries
# ============================================================

qaoa_summary = _quality_summary_from_rows(
    qaoa_quality_rows,
    qaoa_quality,
)

greedy_summary = _quality_summary_from_rows(
    greedy_quality_rows,
    greedy_quality,
)

comparison_metrics = [
    "n_nodes",
    "n_elements",
    "min_angle",
    "mean_min_angle",
    "mean_aspect_ratio",
    "max_aspect_ratio",
    "mean_skewness",
    "max_skewness",
    "mean_ratio_quality",
]


# ============================================================
# Figure 1: Summary table
# ============================================================

fig_table = go.Figure(data=[
    go.Table(
        header=dict(
            values=["Metric", "QAOA", "Greedy"],
            fill_color="#111827",
            font=dict(color="white"),
        ),
        cells=dict(
            values=[
                comparison_metrics,
                [f"{qaoa_summary[m]:.6g}" for m in comparison_metrics],
                [f"{greedy_summary[m]:.6g}" for m in comparison_metrics],
            ],
            fill_color="#f9fafb",
        ),
    )
])

fig_table.update_layout(
    title="QAOA vs Greedy Mesh Quality Summary",
    width=820,
    height=440,
)

fig_table.show()





# ============================================================
# Figure 2: Bar chart comparison
# ============================================================

bar_metrics = [
    "min_angle",
    "mean_min_angle",
    "mean_aspect_ratio",
    "mean_skewness",
    "max_skewness",
    "mean_ratio_quality",
]

fig_bars = go.Figure()

fig_bars.add_trace(go.Bar(
    x=bar_metrics,
    y=[qaoa_summary[m] for m in bar_metrics],
    name="QAOA",
    marker_color="#1f77b4",
))

fig_bars.add_trace(go.Bar(
    x=bar_metrics,
    y=[greedy_summary[m] for m in bar_metrics],
    name="Greedy",
    marker_color="#ff7f0e",
))

fig_bars.update_layout(
    title="Quality Metrics: QAOA vs Greedy",
    xaxis_title="metric",
    yaxis_title="value",
    barmode="group",
    template="plotly_white",
    width=1000,
    height=520,
)

fig_bars.show()





# ============================================================
# Figure 3: Mesh skewness comparison
# ============================================================

qaoa_edge_x, qaoa_edge_y = _mesh_edges_xy(
    qaoa_mesh_nodes,
    qaoa_triangles,
)

greedy_edge_x, greedy_edge_y = _mesh_edges_xy(
    greedy_mesh_nodes,
    greedy_triangles,
)

fig_mesh_compare = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "QAOA mesh skewness",
        "Greedy mesh skewness",
    ),
)

# QAOA mesh edges
fig_mesh_compare.add_trace(
    go.Scattergl(
        x=qaoa_edge_x,
        y=qaoa_edge_y,
        mode="lines",
        line=dict(
            color="rgba(60,60,60,0.35)",
            width=1,
        ),
        showlegend=False,
    ),
    row=1,
    col=1,
)

# QAOA skewness markers
fig_mesh_compare.add_trace(
    go.Scattergl(
        x=qaoa_centroids[:, 0],
        y=qaoa_centroids[:, 1],
        mode="markers",
        marker=dict(
            size=11,
            color=[
                row["skewness"]
                for row in qaoa_quality_rows
            ],
            coloraxis="coloraxis",
        ),
        showlegend=False,
        hovertext=[
            (
                f"tri {row['triangle_id']}"
                f"<br>skewness={row['skewness']:.4f}"
                f"<br>min_angle={row['min_angle']:.2f}"
            )
            for row in qaoa_quality_rows
        ],
        hovertemplate="%{hovertext}<extra></extra>",
    ),
    row=1,
    col=1,
)

# Greedy mesh edges
fig_mesh_compare.add_trace(
    go.Scattergl(
        x=greedy_edge_x,
        y=greedy_edge_y,
        mode="lines",
        line=dict(
            color="rgba(60,60,60,0.35)",
            width=1,
        ),
        showlegend=False,
    ),
    row=1,
    col=2,
)

# Greedy skewness markers
fig_mesh_compare.add_trace(
    go.Scattergl(
        x=greedy_centroids[:, 0],
        y=greedy_centroids[:, 1],
        mode="markers",
        marker=dict(
            size=11,
            color=[
                row["skewness"]
                for row in greedy_quality_rows
            ],
            coloraxis="coloraxis",
        ),
        showlegend=False,
        hovertext=[
            (
                f"tri {row['triangle_id']}"
                f"<br>skewness={row['skewness']:.4f}"
                f"<br>min_angle={row['min_angle']:.2f}"
            )
            for row in greedy_quality_rows
        ],
        hovertemplate="%{hovertext}<extra></extra>",
    ),
    row=1,
    col=2,
)

fig_mesh_compare.update_layout(
    title="Element Skewness Heat Map: QAOA vs Greedy",
    template="plotly_white",
    width=1200,
    height=620,
    coloraxis=dict(
        colorscale="Turbo",
        cmin=0.0,
        cmax=1.0,
        colorbar=dict(title="skewness"),
    ),
)

fig_mesh_compare.update_xaxes(
    title_text="x",
    row=1,
    col=1,
)

fig_mesh_compare.update_xaxes(
    title_text="x",
    row=1,
    col=2,
)

fig_mesh_compare.update_yaxes(
    title_text="y",
    row=1,
    col=1,
)

fig_mesh_compare.update_yaxes(
    title_text="y",
    row=1,
    col=2,
)

fig_mesh_compare.show()





# ============================================================
# Figure 4: Random nodes / patches / merged selection
# ============================================================

nodes = np.asarray(result["nodes"], dtype=float)
patches = result["patches"]
records = result["records"]

merged_indices = np.asarray(
    result["merged_indices"],
    dtype=np.intp,
)

mesh_info = result["mesh_info"]

mesh_nodes = np.asarray(
    mesh_info["nodes"],
    dtype=float,
)

triangles = np.asarray(
    mesh_info["triangles"],
    dtype=np.intp,
)

global_idx = np.asarray(
    mesh_info["global_idx"],
    dtype=np.intp,
)

fig_nodes = go.Figure()

# Random nodes
fig_nodes.add_trace(go.Scattergl(
    x=nodes[:, 0],
    y=nodes[:, 1],
    mode="markers+text",
    marker=dict(
        size=7,
        color="black",
        opacity=0.65,
    ),
    text=[str(i) for i in range(len(nodes))],
    textposition="top center",
    hovertemplate="node %{text}<extra></extra>",
    name="Random nodes",
))

# Patches
for i, patch in enumerate(patches):
    interior = np.asarray(
        patch.get("interior_idx", []),
        dtype=np.intp,
)
    halo = np.asarray(
        patch.get("halo_idx", []),
        dtype=np.intp,
)
    patch_idx = (
        np.concatenate([interior, halo])
        if halo.size
        else interior
    )
    if patch_idx.size == 0:
        continue
    pts = nodes[patch_idx]
    fig_nodes.add_trace(go.Scattergl(
        x=pts[:, 0],
        y=pts[:, 1],
        mode="markers",
        marker=dict(
            size=10,
            opacity=0.28,
        ),
        name=f"patch {i}",
        hovertemplate=f"patch {i}<extra></extra>",
        showlegend=False,
    ))
    center = pts.mean(axis=0)
    radius = float(np.max(np.linalg.norm(pts - center, axis=1)))
    theta = np.linspace(0, 2 * np.pi, 120)
    circle_x = center[0] + radius * np.cos(theta)
    circle_y = center[1] + radius * np.sin(theta)
    fig_nodes.add_trace(go.Scattergl(
        x=circle_x,
        y=circle_y,
        mode="lines",
        line=dict(color="rgba(31,119,180,0.35)", width=1),
        name=f"patch {i} boundary",
        showlegend=False,
        hoverinfo="skip",
    ))

# Merged QAOA nodes
fig_nodes.add_trace(go.Scattergl(
    x=nodes[merged_indices, 0],
    y=nodes[merged_indices, 1],
    mode="markers",
    marker=dict(
        size=12,
        color="#d62728",
        symbol="circle-open",
        line=dict(width=2),
    ),
    name="Merged QAOA nodes",
))

fig_nodes.update_layout(
    title=(
        f"Random 2D Nodes, Patches, and Merged Selection "
        f"(N={len(nodes)}, n_patch={n_patch})"
    ),
    xaxis_title="x",
    yaxis_title="y",
    template="plotly_white",
    width=950,
    height=760,
)

fig_nodes.update_yaxes(
    scaleanchor="x",
    scaleratio=1,
)

fig_nodes.show()






In [226]:

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

visualize_dir = Path.cwd() / "visualise" / timestamp
visualize_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving visualizations to: {visualize_dir}")

SAVE_SCALE = 3

fig_table.write_image(
    visualize_dir / "mesh_quality_summary_table.png",
    scale=SAVE_SCALE,
)

fig_bars.write_image(
    visualize_dir / "quality_metrics_bar_chart.png",
    scale=SAVE_SCALE,
)

fig_mesh_compare.write_image(
    visualize_dir / "mesh_skewness_comparison.png",
    scale=SAVE_SCALE,
)

fig_nodes.write_image(
    visualize_dir / "random_nodes_patches_merged_selection.png",
    scale=SAVE_SCALE,
)


# ============================================================
# Final summary
# ============================================================

print("\nSaved visualization files:")
for path in sorted(visualize_dir.iterdir()):
    print(f" - {path.name}")

{
    "qaoa": qaoa_summary,
    "greedy": greedy_summary,
}


Saving visualizations to: /home/sxm/py_projects/MeshOptimiser/main/visualise/2026-05-20_16-50-21

Saved visualization files:
 - mesh_quality_summary_table.png
 - mesh_skewness_comparison.png
 - quality_metrics_bar_chart.png
 - random_nodes_patches_merged_selection.png


{'qaoa': {'n_nodes': 18.0,
  'n_elements': 27.0,
  'min_angle': 4.631895110482055,
  'mean_min_angle': 31.062973944761964,
  'mean_aspect_ratio': 2.008143880265118,
  'max_aspect_ratio': 3.6764528333679727,
  'mean_skewness': 0.4822837675873006,
  'max_skewness': 0.9228017481586324,
  'mean_ratio_quality': 0.5177162324126995},
 'greedy': {'n_nodes': 18.0,
  'n_elements': 29.0,
  'min_angle': 0.0189771059222601,
  'mean_min_angle': 32.17324089291109,
  'mean_aspect_ratio': 1.7947528761060796,
  'max_aspect_ratio': 3.965570601529101,
  'mean_skewness': 0.46377931845148185,
  'max_skewness': 0.9996837149012957,
  'mean_ratio_quality': 0.5362206815485181}}